# PF00042: complete protein-domain-to-graph pipeline

## Purpose

One labelled protein-domain collection produces one graph per available score
method:

```text
Pfam domain FASTA
    -> Biopython / local BLAST / optional DEDAL scores
    -> canonical pair table and diagnostics
    -> explicit edge rule per method
    -> graph files, tables, summaries, settings, and checksums
```

PF00042 is Pfam's globin family. Pfam builds families from curated seed
alignments and profile hidden Markov models; clans group related families
([Mistry et al., 2021](https://doi.org/10.1093/nar/gkaa913)). Each node is an
ungapped domain instance. Pilot results validate pipeline integration.

Implementations: Biopython exact local alignment
([Cock et al., 2009](https://doi.org/10.1093/bioinformatics/btp163)), local
NCBI BLAST+ heuristic search
([Camacho et al., 2009](https://doi.org/10.1186/1471-2105-10-421)), and released
pretrained DEDAL
([Llinares-López et al., 2023](https://doi.org/10.1038/s41592-022-01700-2)).
Project code supplies orchestration, provenance, and graph analysis.


In [1]:
import json
import subprocess
import sys
from pathlib import Path

import networkx as nx
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATASET_DIRECTORY = PROJECT_ROOT / "data/processed/pfam_pf00042_globin_pilot"
FASTA_PATH = DATASET_DIRECTORY / "PF00042_globin_pilot.fasta"
METADATA_PATH = DATASET_DIRECTORY / "PF00042_globin_pilot_metadata.tsv"
SCORE_DIRECTORY = PROJECT_ROOT / "outputs/tables/pfam_pf00042_correlation"
GRAPH_DIRECTORY = PROJECT_ROOT / "outputs/graphs/pfam_pf00042_pipeline"


## 1. Configure pipeline

`RUN_SCORING=False` loads saved scores. Set `True` to regenerate
`METHODS_TO_RUN`. Biopython and BLAST use main environment; DEDAL uses optional
environment described in `docs/DEDAL.md`.

For available pair set $A_m$ under method $m$, `top_n=N` requires
$0\leq N\leq\lvert A_m\rvert$ and retains first $N$ pairs after descending score order
and deterministic identifier tie-breaking. Equal $N$ gives equal edge budgets.
Notebook 03 evaluates `top_k`, percentile, and target-density
rules. Quality tables report rank correlation, overlap, and coverage; pass/fail
limits remain configurable.


In [2]:
RUN_SCORING = False
METHODS_TO_RUN = ("biopython", "blast")  # add "dedal" only after its optional setup

GRAPH_SPECS = {
    "biopython": {"rule": "top_n", "threshold": 13},
    "blast": {"rule": "top_n", "threshold": 13},
    "dedal": {"rule": "top_n", "threshold": 13},
}

METHOD_SCORE_COLUMNS = {
    "biopython": "biopython_score",
    "blast": "blast_bit_score",
    "dedal": "dedal_sw_score",
}

CORRELATION_CHECKS = {
    "minimum_rho": None,
    "minimum_overlap_fraction": None,
    "minimum_pairs": None,
}


## 2. Load domain collection

FASTA stores identifiers and ungapped amino-acid sequences. Metadata stores
subgroup, organism, taxonomy, source coordinates, and domain length. Selection
algorithm appears in `scripts/prepare_pfam_globin_pilot.py`; exact sources and
checksums appear in dataset manifest under
`data/processed/pfam_pf00042_globin_pilot/`.

Cell calculates sequence and unordered-pair counts from loaded data.


In [3]:
metadata = pd.read_csv(METADATA_PATH, sep="\t")
protein_ids = metadata["uniprot_accession"].tolist()
expected_pair_count = len(protein_ids) * (len(protein_ids) - 1) // 2
print(f"Proteins: {len(protein_ids)}")
print(f"Unique pairs: {expected_pair_count}")
display(metadata[["uniprot_accession", "organism", "subgroup", "domain_length"]])

Proteins: 12
Unique pairs: 66


,uniprot_accession,organism,subgroup,domain_length
0,P02204,Cyprinus carpio (Common carp),myoglobin,114
1,P02200,Alligator mississippiensis (American alligator),myoglobin,117
2,P02206,Heterodontus portusjacksoni (Port Jackson shark),myoglobin,116
3,P02143,Heterodontus portusjacksoni (Port Jackson shark),beta-like haemoglobin,112
4,P02133,Xenopus laevis (African clawed frog),beta-like haemoglobin,117
5,P04443,Mus musculus (Mouse),beta-like haemoglobin,117
6,P02020,Lepidosiren paradoxus (South American lungfish),alpha-like haemoglobin,113
7,P06714,Equus caballus (Horse),alpha-like haemoglobin,111
8,P01967,Bos grunniens (Wild yak) (Bos mutus grunniens),alpha-like haemoglobin,111
9,P09187,Medicago sativa (Alfalfa),divergent globin,116


## 3. Produce or reuse method scores

`scripts/run_pf00042_correlation.py` creates one canonical row per unordered
domain pair and joins methods by pair identifier. `NaN` in BLAST column denotes
an unreported pair at configured threshold; numeric zero retains its ordinary
score meaning. DEDAL contributes learned alignment score and auxiliary homology
logit from one model.

Run manifest records checksums, versions, parameters, runtimes, and outputs.
Cell loads saved pair table after optional regeneration.


In [4]:
if RUN_SCORING:
    scoring_command = [
        sys.executable,
        str(PROJECT_ROOT / "scripts/run_pf00042_correlation.py"),
        "--fasta",
        str(FASTA_PATH),
        "--output-directory",
        str(SCORE_DIRECTORY),
        "--methods",
        *METHODS_TO_RUN,
        "--threads",
        "4",
    ]
    subprocess.run(scoring_command, cwd=PROJECT_ROOT, check=True)

paired_scores = pd.read_csv(SCORE_DIRECTORY / "paired_scores.tsv", sep="\t")
assert len(paired_scores) == expected_pair_count
display(paired_scores.head())

,protein_a,protein_b,biopython_score,blast_raw_score,blast_bit_score,blast_evalue,blast_percent_identity,blast_alignment_length,blast_query_coverage_hsp,dedal_sw_score,...,dedal_gap_count,dedal_alignment_length,biopython_score_rank,biopython_score_percentile,blast_bit_score_rank,blast_bit_score_percentile,dedal_sw_score_rank,dedal_sw_score_percentile,dedal_homology_logit_rank,dedal_homology_logit_percentile
0,P01967,P02020,248.0,257.0,103.0,4.410000e-34,44.248,113.0,100.0,39.242275,...,2,113,3.0,0.969697,3.0,0.96,3.0,0.969697,3.0,0.969697
1,P01967,P02133,205.0,218.0,88.6,4.210000e-28,40.000,110.0,94.0,33.116547,...,6,117,10.0,0.863636,10.0,0.82,6.0,0.924242,6.0,0.924242
2,P01967,P02143,167.0,179.0,73.6,2.980000e-22,38.889,108.0,96.0,28.826246,...,1,112,16.0,0.772727,16.0,0.70,13.0,0.818182,13.0,0.818182
3,P01967,P02200,81.0,NaN,NaN,NaN,NaN,NaN,NaN,16.491179,...,6,117,30.0,0.560606,NaN,NaN,29.0,0.575758,29.0,0.575758
4,P01967,P02204,103.0,97.0,42.0,5.040000e-10,27.193,114.0,96.0,15.432405,...,7,116,27.0,0.606061,27.0,0.48,31.0,0.545455,31.0,0.545455


## 4. Build one graph per method

`scripts/build_method_graphs.py` applies configured edge rule to each available
method column on a common node set. Outputs comprise GraphML, node and edge
tables, structural summary, quality flags, and manifest. Missing optional score
columns produce a recorded skip.


In [5]:
available_graph_specs = {
    method: specification
    for method, specification in GRAPH_SPECS.items()
    if METHOD_SCORE_COLUMNS[method] in paired_scores.columns
}
skipped_methods = sorted(set(GRAPH_SPECS) - set(available_graph_specs))
if skipped_methods:
    print("Skipping methods with no saved score column:", ", ".join(skipped_methods))

graph_command = [
    sys.executable,
    str(PROJECT_ROOT / "scripts/build_method_graphs.py"),
    "--fasta",
    str(FASTA_PATH),
    "--pairs",
    str(SCORE_DIRECTORY / "paired_scores.tsv"),
    "--metadata",
    str(METADATA_PATH),
    "--correlations",
    str(SCORE_DIRECTORY / "spearman_correlations.tsv"),
    "--output-directory",
    str(GRAPH_DIRECTORY),
]
for method, specification in available_graph_specs.items():
    graph_command.extend(
        [
            "--graph",
            f"{method}:{specification['rule']}:{specification['threshold']}",
        ]
    )
for option, value in CORRELATION_CHECKS.items():
    if value is not None:
        graph_command.extend([f"--{option.replace('_', '-')}", str(value)])
subprocess.run(graph_command, cwd=PROJECT_ROOT, check=True)


Built 3 separate graph(s) from 12 proteins
Quality thresholds configured: False
Manifest: /Users/davin/Desktop/Protein Seq Alignment/outputs/graphs/pfam_pf00042_pipeline/graph_run_manifest.json


CompletedProcess(args=['/Users/davin/Desktop/Protein Seq Alignment/.venv/bin/python', '/Users/davin/Desktop/Protein Seq Alignment/scripts/build_method_graphs.py', '--fasta', '/Users/davin/Desktop/Protein Seq Alignment/data/processed/pfam_pf00042_globin_pilot/PF00042_globin_pilot.fasta', '--pairs', '/Users/davin/Desktop/Protein Seq Alignment/outputs/tables/pfam_pf00042_correlation/paired_scores.tsv', '--metadata', '/Users/davin/Desktop/Protein Seq Alignment/data/processed/pfam_pf00042_globin_pilot/PF00042_globin_pilot_metadata.tsv', '--correlations', '/Users/davin/Desktop/Protein Seq Alignment/outputs/tables/pfam_pf00042_correlation/spearman_correlations.tsv', '--output-directory', '/Users/davin/Desktop/Protein Seq Alignment/outputs/graphs/pfam_pf00042_pipeline', '--graph', 'biopython:top_n:13', '--graph', 'blast:top_n:13', '--graph', 'dedal:top_n:13'], returncode=0)

## 5. Verify stored graphs

Cell loads graph and quality summaries, reopens each GraphML file, and checks
that every input domain appears as a node. Edge counts test pipeline execution;
Notebook 03 evaluates scientific graph settings.


In [6]:
summaries = pd.read_csv(GRAPH_DIRECTORY / "graph_summaries.tsv", sep="	")
quality = pd.read_csv(GRAPH_DIRECTORY / "quality_flags.tsv", sep="	")
graphs = {}
for method, specification in available_graph_specs.items():
    label = str(specification["threshold"]).replace(".", "p")
    path = GRAPH_DIRECTORY / f"{method}_{specification['rule']}_{label}.graphml"
    graphs[method] = nx.read_graphml(path)
    assert set(graphs[method]) == set(protein_ids)

display(summaries.loc[summaries["method"].isin(available_graph_specs)])
display(quality)


,method,threshold_rule,threshold,nodes,edges,isolates,connected_components,density,average_clustering,possible_pairs,available_pairs,available_pair_fraction,target_edges,selection_shortfall
0,biopython,top_n,13.0,12,13,3,5,0.19697,0.583333,66,66,1.000000,-1,0
1,blast,top_n,13.0,12,13,3,5,0.19697,0.583333,66,50,0.757576,-1,0
2,dedal,top_n,13.0,12,13,4,6,0.19697,0.450000,66,66,1.000000,-1,0


,score_a,score_b,spearman_rho,n_pairs,overlap_fraction,minimum_rho,minimum_overlap_fraction,minimum_pairs,status,flags
0,biopython_score,blast_bit_score,0.993036,50,0.757576,NaN,NaN,NaN,not_evaluated,NaN
1,biopython_score,dedal_sw_score,0.873681,66,1.000000,NaN,NaN,NaN,not_evaluated,NaN
2,blast_bit_score,dedal_sw_score,0.891749,50,0.757576,NaN,NaN,NaN,not_evaluated,NaN


In [7]:
score_manifest = json.loads((SCORE_DIRECTORY / "run_manifest.json").read_text())
correlations = pd.read_csv(
    SCORE_DIRECTORY / "spearman_correlations.tsv", sep="	"
)

runtime_table = pd.DataFrame(
    [
        {"method": method, "runtime_seconds": runtime}
        for method, runtime in score_manifest["runtimes_seconds"].items()
    ]
).sort_values("method", ignore_index=True)

score_columns = [
    column for column in METHOD_SCORE_COLUMNS.values() if column in paired_scores
]
coverage_table = pd.DataFrame(
    [
        {
            "score_column": column,
            "reported_pairs": int(paired_scores[column].notna().sum()),
            "possible_pairs": len(paired_scores),
            "reported_fraction": paired_scores[column].notna().mean(),
        }
        for column in score_columns
    ]
)

print("Stored method runtimes from the score-run manifest")
display(runtime_table)
print("Observed score availability in the canonical pair table")
display(coverage_table)
print("Pairwise Spearman correlations, calculated only on pairs scored by both methods")
display(correlations)


Stored method runtimes from the score-run manifest


,method,runtime_seconds
0,biopython,0.011717
1,blast,2.737202
2,dedal,313.202586


Observed score availability in the canonical pair table


,score_column,reported_pairs,possible_pairs,reported_fraction
0,biopython_score,66,66,1.000000
1,blast_bit_score,50,66,0.757576
2,dedal_sw_score,66,66,1.000000


Pairwise Spearman correlations, calculated only on pairs scored by both methods


,score_a,score_b,spearman_rho,n_pairs
0,biopython_score,blast_bit_score,0.993036,50
1,biopython_score,dedal_sw_score,0.873681,66
2,biopython_score,dedal_homology_logit,0.873681,66
3,blast_bit_score,dedal_sw_score,0.891749,50
4,blast_bit_score,dedal_homology_logit,0.891749,50
5,dedal_sw_score,dedal_homology_logit,1.000000,66


## Interpretation and provenance

Loaded artifacts verify common pair-table and graph interfaces. Spearman
correlation describes rank agreement over jointly reported pairs; interpret it
with `n_pairs` and method coverage. Equal edge counts here isolate pipeline
behaviour under one fixed budget.

| Question | Stored artifact | Producing code |
|---|---|---|
| Sequences | `data/processed/pfam_pf00042_globin_pilot/*metadata.tsv` and manifest | `scripts/prepare_pfam_globin_pilot.py` |
| Pair scores | `outputs/tables/pfam_pf00042_correlation/paired_scores.tsv` | `scripts/run_pf00042_correlation.py` |
| Parameters, versions, runtimes | `outputs/tables/pfam_pf00042_correlation/run_manifest.json` | same scoring script |
| Rank agreement | `outputs/tables/pfam_pf00042_correlation/spearman_correlations.tsv` | same scoring script |
| Graphs | `outputs/graphs/pfam_pf00042_pipeline/*.graphml` and `*.edges.tsv` | `scripts/build_method_graphs.py` |
| Structural and quality summaries | `graph_summaries.tsv` and `quality_flags.tsv` | same graph script |

Notebook 03 replaces purposive pilot with repeated Pfam-domain collections and
varies composition, method, edge rule, and parameter.

## References

BibTeX records: `references/references.bib`. Keys used here:
`mistry2021pfam`, `cock2009biopython`, `camacho2009blastplus`,
`llinareslopez2023deep`, `smith1981identification`, `gotoh1982improved`, and
`henikoff1992amino`.
